In [81]:
import json
import os
from tqdm import tqdm

In [82]:
def load_json(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        return json.load(file)

def save_json(data, file_path):
    with open(file_path, 'w', encoding='utf-8') as file:
        json.dump(data, file, ensure_ascii=False, indent=4)

In [83]:
import re

def get_number_from_string(text):
    # Tìm dấu gạch dưới và các chữ số ở cuối chuỗi
    match = re.search(r'_(\d+)$', text)
    if match:
        return int(match.group(1)) # Trả về số dưới dạng string
    return None

# Test thử
data = ["WIKI_10", "CRAWL_NER_WIKI_25", "VERSION_99"]
results = [get_number_from_string(s) for s in data]
print(results) 

[10, 25, 99]


In [84]:
CACHE_URL_PATH = r"data\cache\mapping\hash2url.json"
CACHE_SEARCH_PATH = r"data\cache\mapping\hash_id_IR_20.json"
METADATA_PATH = r"data\metadata_medhop.json"

BASE_1000 = "data/ALL_ROUND/medhop_results_gg_direct"
DIR_SEARCH = "data/cache/IR_20"
DIR_WIKI = "data/cache/wiki_pages_tool_1"

In [85]:
def flatten_wiki_cache(file_path):
    # 1. Load dữ liệu từ file
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            raw_data = json.load(f)
    except Exception as e:
        print(f"Lỗi khi đọc file: {e}")
        return {}

    final_cache = {}

    # 2. Duyệt qua từng cấp độ của JSON
    # main_id: "157829"
    # tasks: {"crawl_wiki": {...}, "crawl_ner_wiki": {...}}
    for main_id, tasks in raw_data.items():
        for task_name, items in tasks.items():
            # Duyệt qua các cặp hash_id: url trong từng task
            for hash_id, url in items.items():
                # Lưu vào dict chung
                final_cache[hash_id] = url

    return final_cache

FILE_PATH = r'data\cache\all_hash_id_crawl_wiki_embed.json' 
url_mapping = flatten_wiki_cache(FILE_PATH)

print(f"Tổng số HashID thu được: {len(url_mapping)}")

Tổng số HashID thu được: 4074


In [ ]:
CACHE_URL_WIKI = load_json(CACHE_URL_PATH)
CACHE_SEARCH = load_json(CACHE_SEARCH_PATH)
metadata = load_json(METADATA_PATH)
    
CACHE_URL_WIKI.update(url_mapping)

# SEARCH_MAPPING = load_json(r'data\cache\mapping\hash_id copy.json')
# CACHE_SEARCH.update(SEARCH_MAPPING)
# CACHE_SEARCH.update(load_json(r'data\cache\mapping\hash_id.json'))

# save_json(CACHE_SEARCH, CACHE_SEARCH_PATH)

In [87]:
len(CACHE_URL_WIKI)

53017

In [88]:
import re

def load_retrieval_map(data_input):
    """
    Map ID (WIKI_10, CRAWL_NER_WIKI_2,...) trực tiếp với hash_id theo thứ tự 1-1.
    """
    result_map = {}
    
    for key, entry in data_input.items():
        crawl_docs = entry.get("crawl_doc", [])
        hash_ids = entry.get("hash_id", [])
        
        for i, doc_text in enumerate(crawl_docs):
            # Regex mới: 
            # \[ : Bắt đầu ngoặc vuông
            # ( : Bắt đầu group muốn lấy
            #  .*? : Mọi ký tự (WIKI, CRAWL_NER_WIKI,...)
            #  _ : Dấu gạch dưới
            #  \d+ : Các chữ số
            # ) : Kết thúc group
            # \] : Đóng ngoặc vuông
            match = re.search(r'\[(.*?_\d+)\]', doc_text)
            
            if match and i < len(hash_ids):
                source_key = match.group(1) # Kết quả: "WIKI_10" hoặc "CRAWL_NER_WIKI_2"
                result_map[source_key] = hash_ids[i]
                
    return result_map

In [89]:
def extract_detail_regex(text):
    pattern = r'(?i)Detailed explanation[:\s]*(.*)'
    match = re.search(pattern, text)
    
    if match:
        return match.group(1).strip()
    return ""

long_answer = "Reference: [GG_5], [GG_6]. Detailed explanation: The oxygen level..."
print(extract_detail_regex(long_answer))

The oxygen level...


```Question: 
===
Subquestion 1: 
Question:
Answer:
Referece:
Detailed explanation:

Subquestion 2:
Question:
Answer:
Referece:
Detailed explanation:
===
Conclusion:
Answer:
Detailed explanation: 
=============
Question:
Answer:
Referece:
Detailed explanation: ```

# DIRECT

In [90]:
BASE_1000 = "data/ALL_ROUND/medhop_results_gg_direct/all_ans"
DIR_SEARCH = "data/cache/IR_20"
DIR_WIKI = "data/cache/wiki_pages_tool_1"

In [96]:
output_direct = {}
for qidx, item in metadata.items():
    if item['type_q'] != 'direct':
        continue
    
    file_path = f"{BASE_1000}/R4_step_3b_all_ans_{qidx}.json"
    tmp_data = load_json(file_path)
    
    final_answer = tmp_data['1']['0']['0']['final_answer']
    question = tmp_data['1']['0']['0']['question']
    search_kw = tmp_data['1']['0']['0']['search_keyword'].lower().strip()
    search_kw_raw = search_kw + " "
    short_answer = final_answer['short_answer']
    long_answer = final_answer['long_answer']
    
    reference_inf = f"Question: {question}\n Answer: {short_answer}\n Referece:\n "
    
    if final_answer['selection_status'] != 'selected':
        reference_inf += "Zeroshot - O3" + " \n "
        detail_explanation = long_answer
    else:
        detail_explanation = extract_detail_regex(long_answer)
        sources = final_answer['sources']
        
        retrieval_information = tmp_data['1']['0']['0']['retrieval_information']
        
        if search_kw in CACHE_SEARCH or search_kw_raw in CACHE_SEARCH:
            if search_kw in CACHE_SEARCH:
                hash_id_gg = CACHE_SEARCH[search_kw]
            else:
                hash_id_gg = CACHE_SEARCH[search_kw_raw]
                
            if '.json' not in hash_id_gg:
                hash_id_gg = f"{hash_id_gg}.json"
            file_gg = f'{DIR_SEARCH}/{hash_id_gg}'
            gg_data = load_json(file_gg)['google']['search_results']
        else:
            print(f"CACHE 0 co SEARCH GG  {qidx}")
            gg_data = None
            # continue
        
        crawl_wiki_data = load_retrieval_map(retrieval_information)
        
        for src in sources:
            index_link = get_number_from_string(src)
            if index_link:
                index_link = index_link -  1 # start = 0
            else:            
                continue
                    
            if 'GG' in src:            
                reference_inf += gg_data[index_link]['link'] + "\n "
            else:
                if src not in crawl_wiki_data:
                    print(qidx)
                    print(file_path)
                    print(crawl_wiki_data)
                    print(sources)
                    continue
                
                wiki_hashid = crawl_wiki_data[src]
                file_wiki = f"{DIR_WIKI}/{wiki_hashid}"
                print(file_wiki)
                # if os.path.exists()
                reference_inf += CACHE_URL_WIKI[wiki_hashid] + "\n "   
            
    reference_inf += f"Detailed explanation: {detail_explanation}"
        
    print(reference_inf)
    output_direct[qidx] = reference_inf
    

Question: What rare congenital multiple-anomaly syndrome is caused by a mutation in the SKI gene on chromosome 1 and is characterized by abnormally long, slender fingers referred to as 'spider fingers'?
 Answer: shprintzen-goldberg syndrome
 Referece:
 https://en.wikipedia.org/wiki/Shprintzen%E2%80%93Goldberg_syndrome
 https://my.clevelandclinic.org/health/diseases/24054-shprintzen-goldberg-syndrome
 https://medlineplus.gov/genetics/condition/shprintzen-goldberg-syndrome/
 Detailed explanation: The context includes multiple references to Shprintzen-Goldberg syndrome, a rare congenital multiple-anomaly syndrome. It is known by other names such as marfanoid-craniosynostosis syndrome and is characterized by skeletal abnormalities including features like arachnodactyly (spider fingers). Although the text does not explicitly state the genetic mutation, Shprintzen-Goldberg syndrome is associated with a mutation in the SKI gene on chromosome 1.
Question: What protein, encoded by a gene locate

In [97]:
save_json(output_direct, 'data/EVIDENCE/reference_direct_1000.json')
print(len(output_direct.keys()))

704


# SEQUENCE

In [99]:
output_seq = {}
BASE_SEQ = "SEQ_medhop_results/all_ans"
seq_combine = load_json(r"data\ALL_ROUND\1000\DECOMP_NEW_medhop_results\combine\seq_combine_zeroshot.json")

for qidx, item in metadata.items():
    qidx = str(qidx)
    if item['type_q'] != 'sequence':
        continue
    
    # if qidx == "442406": #"116951":
    #     continue
    
    file_path = f"{BASE_SEQ}/R4_step_3b_all_ans_{qidx}.json"
    tmp_data = load_json(file_path)
    reference_inf = ""
    prev_answer = ""
    for step, step_data in tmp_data['1']['0'].items():
        reference_inf += f"Subquestion {int(step)+1}:\n"
        step = str(step)
        print("STEP: ", step)
        final_answer = tmp_data['1']['0'][step]['final_answer']
        question = tmp_data['1']['0'][step]['question']
        search_kw = tmp_data['1']['0'][step]['search_keyword']
        
        short_answer = final_answer['short_answer']
        long_answer = final_answer['long_answer']
        
        if step != '0':
            search_kw = search_kw + " " + prev_answer
            
        search_kw = search_kw.lower().strip()
        search_kw_raw = search_kw + " "
        # print(step, search_kw)
        prev_answer = short_answer.lower()
        reference_inf += f"Question: {question}\n Answer: {short_answer}\n Referece:\n "
        
        if final_answer['selection_status'] != 'selected':
            reference_inf += "ZEROSHOT - O3"
        else:        
            detail_explanation = extract_detail_regex(long_answer)
            sources = final_answer['sources']
            
            retrieval_information = tmp_data['1']['0'][step]['retrieval_information']
            crawl_wiki_data = load_retrieval_map(retrieval_information)
            
            # print(search_kw)
            
            if search_kw in CACHE_SEARCH or search_kw_raw in CACHE_SEARCH:
                if search_kw in CACHE_SEARCH:
                    hash_id_gg = CACHE_SEARCH[search_kw]
                else:
                    hash_id_gg = CACHE_SEARCH[search_kw_raw]
                if '.json' not in hash_id_gg:
                    hash_id_gg = f"{hash_id_gg}.json"
                file_gg = f'{DIR_SEARCH}/{hash_id_gg}'
                gg_data = load_json(file_gg)['google']['search_results']
            else:
                print(f"CACHE không co SEARCH GG  {qidx}")
                print(search_kw)
                gg_data = None
                continue
            # print(crawl_wiki_data)
            # print(sources)
            for src in sources:
                index_link = get_number_from_string(src)
                if index_link:
                    index_link = index_link-  1 # start = 0
                else:
                    # print(qidx)
                    # print(file_path)
                    # exit()
                    continue
                
                if 'GG' in src:            
                    reference_inf += gg_data[index_link]['link'] + "\n"
                else:
                    if src not in crawl_wiki_data:
                        continue
                    wiki_hashid = crawl_wiki_data[src]
                    file_wiki = f"{DIR_WIKI}/{wiki_hashid}"
                    print(file_wiki)
                    # if os.path.exists()
                    reference_inf += CACHE_URL_WIKI[wiki_hashid] + "\n" 
                    reference_inf += f"Detailed explanation: {detail_explanation}\n"
            print(reference_inf)        
            
    reference_inf += "================\nConclusion:\n"
    
    if qidx not in seq_combine:
        print(qidx)
        continue   
    
    reference_inf += f"Answer: {seq_combine[qidx]['short_answer']}\n"
    reference_inf += f"Detailed explanation: {seq_combine[qidx]['long_answer']}"
    
    output_seq[qidx] = reference_inf
    
save_json(output_seq, 'data/EVIDENCE/reference_sequence_1000.json')
print(len(output_seq.keys()))

STEP:  0
data/cache/wiki_pages_tool_1/453bbd1fa3b69bcd38acaab40f969c7d.json
Subquestion 1:
Question: Which gene is associated with the inherited disorder characterized by atresia occurring at multiple locations throughout the small and large intestines?
 Answer: ttc7a
 Referece:
 https://www.bdi.ox.ac.uk/publications/2062529
https://en.wikipedia.org/wiki/Intestinal_atresia
https://en.wikipedia.org/wiki/Intestinal_atresia
https://en.wikipedia.org/wiki/familial_multiple_intestinal_atresia
Detailed explanation: The context describes familial multiple intestinal atresia as an inherited disorder where atresia occurs at multiple locations throughout the small and large intestines. It specifically states that the underlying lesion in this condition is caused by a mutation in the TTC7A gene, which is located on the short arm of chromosome 2 (2p16).Reference: [CRAWL_WIKI_1]. Detailed explanation: Familial multiple intestinal atresia is an inherited disorder characterized by atresia at multiple 

In [100]:
print(len(output_seq.keys()))

225


# PARALLEL

In [103]:
output_parallel = {}
BASE_PAR = "data/ALL_ROUND/1000/medhop_results_ct6_r36_parallel_seq/all_ans"
parallel_combine = load_json(r'data\ALL_ROUND\1000\medhop_results_ct6_r36_parallel_seq\parallel_combine.json')
for qidx, item in metadata.items():
    if item['type_q'] != 'parallel':
        continue
    
    # if qidx != "112756":
    #     continue
    
    file_path = f"{BASE_PAR}/R4_step_3b_all_ans_{qidx}.json"
    tmp_data = load_json(file_path)
    reference_inf = ""
    for qi in ["1", "2", "3", "4"]:
        if qi not in tmp_data:
            break
        
        reference_inf += f"Subquestion {qi}:\n"        
        prev_answer = ""
        for step, step_data in tmp_data[qi]['0'].items():
            step = str(step)
            final_answer = tmp_data[qi]['0'][step]['final_answer']
            question = tmp_data[qi]['0'][step]['question']
            search_kw = tmp_data[qi]['0'][step]['search_keyword'].lower().strip()
            
            short_answer = final_answer['short_answer']
            long_answer = final_answer['long_answer']
            
            if step != '0':
                search_kw = search_kw + " " + prev_answer
            search_kw_raw = search_kw + " "
            reference_inf += f"Question: {question}\n Answer: {short_answer}\n Referece:\n "
            
            if final_answer['selection_status'] != 'selected':
                reference_inf += "ZEROSHOT - O3 \n"
            else:        
                detail_explanation = extract_detail_regex(long_answer)
                sources = final_answer['sources']
                
                retrieval_information = tmp_data[qi]['0'][step]['retrieval_information']
                crawl_wiki_data = load_retrieval_map(retrieval_information)
                
                # print(search_kw)
                
                if search_kw in CACHE_SEARCH or search_kw_raw in CACHE_SEARCH:
                    if search_kw in CACHE_SEARCH:
                        hash_id_gg = CACHE_SEARCH[search_kw]
                    else:
                        hash_id_gg = CACHE_SEARCH[search_kw_raw]
                    file_gg = f'{DIR_SEARCH}/{hash_id_gg}'
                else:
                    print(f"CACHE 0 co SEARCH GG  {qidx}")
                    continue
                
                gg_data = load_json(file_gg)['google']['search_results']
                
            
                for src in sources:
                    index_link = get_number_from_string(src)
                    if index_link:
                        index_link = index_link-  1 # start = 0
                    else:
                        continue
                    
                    if 'GG' in src:            
                        reference_inf += gg_data[index_link]['link'] + "\n "
                    else:
                        if src not in crawl_wiki_data:
                            continue
                        wiki_hashid = crawl_wiki_data[src]
                        file_wiki = f"{DIR_WIKI}/{wiki_hashid}"
                        # print(file_wiki)
                        # if os.path.exists()
                        reference_inf += CACHE_URL_WIKI[wiki_hashid] + "\n "   
                        
                reference_inf += f"Detailed explanation: {detail_explanation}"
                    
            prev_answer = short_answer.lower()
            
    reference_inf += "================\nConclusion:\n"
    
    if qidx not in parallel_combine:
        print(qidx)
        continue   
    
    reference_inf += f"Answer: {parallel_combine[qidx]['short_answer']}\n"
    reference_inf += f"Detailed explanation: {parallel_combine[qidx]['long_answer']}"
    
    
    output_parallel[qidx] = reference_inf
    print(reference_inf)
    
save_json(output_parallel, 'data/EVIDENCE/reference_parallel_1000.json')
print(len(output_parallel.keys()))

Subquestion 1:
Question: What chromosomal abnormalities are associated with Absent tibia-polydactyly-arachnoid cyst syndrome?
 Answer: yes
 Referece:
 ZEROSHOT - O3 
Subquestion 2:
Question: What chromosomal abnormalities are associated with Charcot-Marie-Tooth disease?
 Answer: duplication of the pmp22 on chromosome 17
 Referece:
 https://en.wikipedia.org/wiki/chromosome_abnormality
 https://en.wikipedia.org/wiki/absent_tibia-polydactyly-arachnoid_cyst_syndrome
 Detailed explanation: The provided context specifies that Charcot–Marie–Tooth disease type 1A can be caused by a duplication of the gene encoding peripheral myelin protein 22 (PMP22) on chromosome 17. This chromosomal abnormality results in an extra copy of the PMP22 gene, which is directly associated with the disease.================
Conclusion:
Answer: no
Detailed explanation: Explanation: The provided context indicates that Absent tibia-polydactyly-arachnoid cyst syndrome is associated with specific chromosomal deletions (s